In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name_or_path = "TheBloke/Mistral-7B-codealpaca-lora-GPTQ"
# To use a different branch, change revision
# For example: revision="gptq-4bit-32g-actorder_True"
model = AutoModelForCausalLM.from_pretrained(model_name_or_path,
                                             device_map="auto",
                                             trust_remote_code=False,
                                             revision="main")

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=True)



In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    top_k=40,
    repetition_penalty=1.1
)

In [8]:
prompt = "Write a SPARQL code for the following natural language quesiont that run on DBpedia: What is the capital of Germany?"
prompt_template=f'''Below is an instruction that describes a task. Write a response that appropriately completes the request. please encolse the code in ``` and only show code not the detail

### Instruction:
{prompt}

### Response:
'''



print(pipe(prompt_template)[0]['generated_text'])


Below is an instruction that describes a task. Write a response that appropriately completes the request. please encolse the code in ``` and only show code not the detail

### Instruction:
Write a SPARQL code for the following natural language quesiont that run on DBpedia: What is the capital of Germany?

### Response:
 Here's the SPARQL query to get the capital of Germany from DBpedia:

```sparql
PREFIX dbpedia_property_10000041 AS "http://dbpedia.org/ontology/ property/capital"
PREFIX dbpedia_object AS "http://dbpedia.org/ontology/ Germany"
SELECT?capital
WHERE {
 ?x dbo:isMeronym?dbo:country.
 ?x dbpedia_property_10000041?capital.
 ?x dbo:isMeronym?dbo:country.
 ?y dbo:wikiPage?dbo:country.
 ?y dbo:wikiLink?y.
 ?y dbo:wikiLink?x.
  SELECT?capital
  WHERE {
   ?y dbo:hasDbID?dbo:WIKI_PAGE_ID.
   ?y dbo:hasDbID?dbo:WIKI_PAGE_ID.
    FILTER(ABS(STRFIND(str(?y), "wikipedia") - str(?y)) = 0)
  }
}
```

This code uses DBpedia's ontology to find the object with the property "capital", whic

In [57]:
def generate_sparql(question): 
    prompt = "generate a sparql query for the input question:" + str(question)
    prompt_template=f'''Below is an instruction that describes a task. Write a response that appropriately completes the request. please encolse the code in ``` and only show code not the detail

    ### Instruction:
    {prompt}

    ### Response:
    '''
    response = pipe(prompt_template)[0]['generated_text']
    
    if '### Response:' in response:
        s = response.split('### Response:')[1]
        if "```" in s:
            return s.split('```')[1]
        else:
            return s
    else:
        return response

In [58]:
generate_sparql("What is the capital of Pakistan?")

/upb/users/m/manzoor/profiles/unix/cs/.conda/envs/sparqlgen/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


'\n    PREFIX dbo: <http://dbpedia.org/ontology/>\n    SELECT?capital \n    WHERE {\n       ?country dbo:countryType "Pakistan".\n       ?country dbo:capital?capital \n    }\n    '

In [44]:
import pandas as pd
import time

### Experiment 01 on misteral code for wikidata LCQuad

In [23]:
df_lcquad = pd.read_csv('Output/exp1_prompt_output_lcquad.csv')

In [45]:
len(df_lcquad)

5969

In [46]:
df_lcquad.columns

Index(['Unnamed: 0', 'questions', 'sparql_lcquad', 'output'], dtype='object')

In [47]:
start_time = time.time()
df_lcquad['mistral_sparql'] = df_lcquad["questions"].apply(generate_sparql)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time} seconds")

Execution time: 21951.621829986572 seconds


In [105]:
import nltk
from nltk.translate.bleu_score import sentence_bleu

#nltk.download('punkt')

for index, row in df_lcquad.iterrows():
    
    reference = nltk.word_tokenize(row['sparql_lcquad'].lower())
    if len(row['mistral_sparql']) < 1:
        print(index)
        continue
    candidate = nltk.word_tokenize(row['mistral_sparql'].lower())
    bleu_score = sentence_bleu([reference], candidate[1:])
    df_lcquad.at[index, 'Org_bleu_score'] = bleu_score

In [106]:
average_bleu = df_lcquad['Org_bleu_score'].mean()
average_bleu

0.008710084499626892

In [109]:
df_lcquad.to_csv('Output/exp1_mistrel_lcquad.csv')

### Experiment 01 on misteral code for dbpedia qald

In [54]:
df_qald = pd.read_csv('Output/exp1_prompt_output_qald.csv')

In [55]:
len(df_qald)

150

In [56]:
df_qald.head()

,Unnamed: 0,id,questions,sparql_qald,sparql_gen,output
0,0,1,what is the time zone of salt lake city?,PREFIX res: <http://dbpedia.org/resource/> PRE...,SELECT DISTINCT ?uri WHERE { res:Salt_Lake_Cit...,"[INST] \nYou are given an input question, conv..."
1,1,2,who killed caesar?,PREFIX dct: <http://purl.org/dc/terms/> PREFIX...,SELECT DISTINCT ?uri WHERE { ?uri dct:subject ...,"[INST] \nYou are given an input question, conv..."
2,2,3,what is the highest mountain in germany?,PREFIX rdfs: <http://www.w3.org/2000/01/rdf-sc...,SELECT ?uri WHERE { ?uri rdf:type onto:Mountai...,"[INST] \nYou are given an input question, conv..."
3,3,4,which american presidents were in office durin...,PREFIX dbo: <http://dbpedia.org/ontology/> PRE...,SELECT ?uri WHERE { ?uri dct:subject dbc:Presi...,"[INST] \nYou are given an input question, conv..."
4,4,5,butch otter is the governor of which u.s. state?,SELECT DISTINCT ?uri WHERE { ?uri a <http://db...,SELECT DISTINCT ?uri WHERE { ?uri a yago:Wikic...,"[INST] \nYou are given an input question, conv..."


In [59]:
start_time = time.time()
df_qald['mistral_sparql'] = df_qald["questions"].apply(generate_sparql)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time} seconds")

Execution time: 577.8442919254303 seconds


In [100]:


for index, row in df_qald.iterrows():
    
    reference = nltk.word_tokenize(row['sparql_qald'].lower())
    if len(row['mistral_sparql']) < 1:
        print(index)
        continue
    candidate = nltk.word_tokenize(row['mistral_sparql'].lower())
    bleu_score = sentence_bleu([reference], candidate[1:])
    df_qald.at[index, 'Org_bleu_score'] = bleu_score

In [101]:
average_bleu = df_qald['Org_bleu_score'].mean()
average_bleu

0.045427155815744

In [62]:
df_qald.head()

,Unnamed: 0,id,questions,sparql_qald,sparql_gen,output,mistral_sparql,Org_bleu_score
0,0,1,what is the time zone of salt lake city?,PREFIX res: <http://dbpedia.org/resource/> PRE...,SELECT DISTINCT ?uri WHERE { res:Salt_Lake_Cit...,"[INST] \nYou are given an input question, conv...",\n PREFIX dbdp: <http://dbpedia.org/dataset...,0.0
1,1,2,who killed caesar?,PREFIX dct: <http://purl.org/dc/terms/> PREFIX...,SELECT DISTINCT ?uri WHERE { ?uri dct:subject ...,"[INST] \nYou are given an input question, conv...",\n Prefixes: \n dbo: <http://dbpedia.o...,0.0
2,2,3,what is the highest mountain in germany?,PREFIX rdfs: <http://www.w3.org/2000/01/rdf-sc...,SELECT ?uri WHERE { ?uri rdf:type onto:Mountai...,"[INST] \nYou are given an input question, conv...",\n PREFIX dbo: <http://dbpedia.org/onto>\n ...,0.0
3,3,4,which american presidents were in office durin...,PREFIX dbo: <http://dbpedia.org/ontology/> PRE...,SELECT ?uri WHERE { ?uri dct:subject dbc:Presi...,"[INST] \nYou are given an input question, conv...",sparql\n PREFIX dbdw WHERE {\n SELECT DI...,0.0
4,4,5,butch otter is the governor of which u.s. state?,SELECT DISTINCT ?uri WHERE { ?uri a <http://db...,SELECT DISTINCT ?uri WHERE { ?uri a yago:Wikic...,"[INST] \nYou are given an input question, conv...",\n PREFIX dbo <http://dbpedia.org/ontology/...,0.0


In [108]:
df_qald.to_csv('Output/exp1_mistrel_qald.csv')

### Experiment 01 on misteral code for dbpedia Vquanda

In [64]:
df_vquanda = pd.read_csv('Output/exp1_prompt_output_vquanda.csv')

In [65]:
len(df_vquanda)

1000

In [66]:
df_vquanda.head(2)

,Unnamed: 0,questions,sparql_vquanda,output
0,0,Count the number of people became famous for w...,SELECT DISTINCT COUNT(?uri) WHERE { ?x <http:/...,"[INST] \nYou are given an input question, conv..."
1,1,Which location city of Denver Broncos is the p...,SELECT DISTINCT ?uri WHERE { <http://dbpedia.o...,"[INST] \nYou are given an input question, conv..."


In [67]:
start_time = time.time()
df_vquanda['mistral_sparql'] = df_vquanda["questions"].apply(generate_sparql)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time} seconds")

/upb/users/m/manzoor/profiles/unix/cs/.conda/envs/sparqlgen/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


Execution time: 4327.619374036789 seconds


In [98]:
for index, row in df_vquanda.iterrows():
    
    
    reference = nltk.word_tokenize(row['sparql_vquanda'].lower())
    
    if len(row['mistral_sparql']) < 1:
        print(index)
        continue
    candidate = nltk.word_tokenize(row['mistral_sparql'].lower())
    
    bleu_score = sentence_bleu([reference], candidate[1:])
    df_vquanda.at[index, 'Org_bleu_score'] = bleu_score
    

In [99]:
average_bleu = df_vquanda['Org_bleu_score'].mean()
average_bleu

0.005678455490788355

In [107]:
df_vquanda.to_csv('Output/exp1_misteral_vquanda.csv')

### Experiments on GammaCode 

In [81]:

import nltk
from nltk.translate.bleu_score import sentence_bleu

# Example reference and candidate sentences
reference = df_vquanda['sparql_vquanda'].iloc[0]


BLEU score: 0.1370643416731807


In [82]:
df_vquanda['sparql_vquanda'].iloc[0]

'SELECT DISTINCT COUNT(?uri) WHERE { ?x <http://dbpedia.org/ontology/commander> <http://dbpedia.org/resource/Andrew_Jackson> . ?uri <http://dbpedia.org/ontology/knownFor> ?x  . }'

In [78]:
df_vquanda['mistral_sparql'].iloc[0]

'\n    Prefix dbo: <http://dbpedia.org/ontology/dbpedia-base+v2.0/>\n    Prefix dbo: (DBPedia ontology prefix)\n    Prefix dbp: <http://dbpedia.org/ontology/data/dbp/> \n\n    SELECT COUNT(distinct dbo:subject) as Number_of_People\n    WHERE {\n   ?person dbo:hasOccupation?occupation.\n   ?person dbo:role?role.\n   ?person dbo:birthPlace?place.\n   ?person dbo:alive?date.\n   ?person dbo:hasCommander?commander.\n   ?occupation dbo:isSubclassOf?type.\n   ?occupation dbo:isAmbiguous 0.\n   ?type dbo:isSubclassOf <http://dbpedia.org/ontology/dbpedia-base+v2.0/person>.\n   ?commander wdt:itemType?type.\n   ?commander dbo:influence?numberOfYears.\n   ?commander dbo:started?date.\n   ?commander dbo:ended?date.\n   ?commander dbo:militaryRank?rank.\n   ?commander dbo:country?country.\n   ?commander dbo:birthPlace?place.\n   ?commander dbo:name?name.\n    FILTER(str(?commander) = "Andrew Jackson" && str(?date) <= str(date(now))){\n    }\n    }\n    '

In [84]:
df_vquanda['Org_bleu_score'].iloc[0]

0.0

In [89]:
import nltk
from nltk.translate.bleu_score import sentence_bleu

# Example reference and candidate sentences
reference = nltk.word_tokenize(df_vquanda['sparql_vquanda'].iloc[0].lower())
candidate = nltk.word_tokenize(df_vquanda['mistral_sparql'].iloc[0].lower())

# Calculate BLEU score
bleu_score = sentence_bleu([reference], candidate)

print("BLEU score:", bleu_score)


BLEU score: 1.459722779833038e-78


In [ ]:
BLEU score: 1.459722779833038e-78